Dataset REDUCIDO RANDOM FOREST SMOTE

In [ ]:
import time
from imblearn.over_sampling import SMOTE
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split

df_4 = pd.read_csv("/content/dataset3_reducido_pesos.csv")

X = df_4.drop(columns=["burnout_score", "peso"])
y = df_4["burnout_score"]

In [ ]:
t_inicio = time.time()
#=============================================================================
# 1. DIVISIÓN DEL CONJUNTO DE DATOS
# =============================================================================
X_encoded = pd.get_dummies(X, drop_first=True)

# Separación de los datos reales puros (TEST) antes de cualquier balanceo
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.30, random_state=42, stratify=y)

# =============================================================================
# BALANCEO DE CLASES SOLO EN EL CONJUNTO DE ENTRENAMIENTO (SMOTE)
# =============================================================================
smote = SMOTE(random_state=42)
X_train_balanceado, y_train_balanceado = smote.fit_resample(X_train, y_train)

# =============================================================================
# 2. CONFIGURACIÓN DEL ESPACIO DE BÚSQUEDA DE HIPERPARÁMETROS (RANDOM FOREST)
# =============================================================================
param_grid = {
    'n_estimators': [80, 120, 150],       # Acotado para evitar demoras excesivas en 1M de filas
    'max_depth': [6, 10, 14],             # Evita árboles demasiado profundos y lentos
    'min_samples_split': [5, 10],         # Divisiones más estables
    'min_samples_leaf': [2, 5],           # Hojas mínimas para reducir coste y sobreajuste
    'max_features': ['sqrt', 'log2'],     # Menos variables evaluadas por división = más rápido
    'bootstrap': [True]                   # Muestreo con reemplazo para cada árbol
}

print("=== Iniciando Búsqueda Aleatoria de Hiperparámetros (Random Forest) ===")

# Modelo base vacío
rf_base = RandomForestClassifier(
    random_state=42
)

# Configuración del optimizador
random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=6,            # 6 combinaciones x 2 pliegues = 12 ajustes (rápido para calibrar)
    scoring='accuracy',  # Métrica guía para elegir el mejor modelo
    cv=2,                # 2 pliegues para reducir el tiempo total en Colab
    verbose=2,           # Muestra en consola el avance de cada ajuste
    random_state=42,
    n_jobs=-1            # Usa todos los núcleos disponibles del procesador
)

# =============================================================================
# 3. ENTRENAMIENTO Y BÚSQUEDA
# =============================================================================
random_search.fit(X_train_balanceado, y_train_balanceado)
print(f"\nBúsqueda finalizada en {(time.time() - t_inicio) / 60:.2f} minutos.")

# Extraemos los índices de los 3 mejores modelos según el ranking en validación cruzada
resultados_cv = pd.DataFrame(random_search.cv_results_)
top_3_indices = resultados_cv.sort_values(by='rank_test_score').index[:3]

print("\nTOP 3 MEJORES MODELOS ENCONTRADOS:\n")

# =============================================================================
# 4. PREDICCIONES FINALES Y CÁLCULO DE MÉTRICAS
# =============================================================================
for rank, idx in enumerate(top_3_indices, start=1):
    params_top = resultados_cv.loc[idx, 'params']
    cv_score_top = resultados_cv.loc[idx, 'mean_test_score']

    modelo_top = RandomForestClassifier(random_state=42, n_jobs=-1, **params_top)
    modelo_top.fit(X_train_balanceado, y_train_balanceado)

    y_pred_train_top = modelo_top.predict(X_train_balanceado)
    y_pred_test_top = modelo_top.predict(X_test)

    # TRAIN
    accuracy_train = accuracy_score(y_train_balanceado, y_pred_train_top)
    precision_train = precision_score(y_train_balanceado, y_pred_train_top)
    recall_train = recall_score(y_train_balanceado, y_pred_train_top)
    f1_train = f1_score(y_train_balanceado, y_pred_train_top)

    # TEST
    accuracy_test = accuracy_score(y_test, y_pred_test_top)
    precision_test = precision_score(y_test, y_pred_test_top)
    recall_test = recall_score(y_test, y_pred_test_top)
    f1_test = f1_score(y_test, y_pred_test_top)

    # =========================================================================
    # 5. PROYECCIÓN DE RESULTADOS
    # =========================================================================
    print(f"\n--- TOP {rank} | CV Score: {cv_score_top:.4f} ---")
    print(f"Hiperparámetros: {params_top}")
    print("Resultados del Modelo RANDOM FOREST (COMPLETO - SMOTE)")
    print(f"Métrica      |      TRAIN     |     TEST   ")
    print(f"-------------|----------------|---------------")
    print(f"Exactitud    |     {accuracy_train:.4f}     |    {accuracy_test:.4f}")
    print(f"Precisión    |     {precision_train:.4f}     |    {precision_test:.4f}")
    print(f"Sensibilidad |     {recall_train:.4f}     |    {recall_test:.4f}")
    print(f"F1-Score     |     {f1_train:.4f}     |    {f1_test:.4f}")

=== Iniciando Búsqueda Aleatoria de Hiperparámetros (Random Forest) ===
Fitting 2 folds for each of 6 candidates, totalling 12 fits

Búsqueda finalizada en 86.55 minutos.

TOP 3 MEJORES MODELOS ENCONTRADOS:


--- TOP 1 | CV Score: 0.8993 ---
Hiperparámetros: {'n_estimators': 150, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 14, 'bootstrap': True}
Resultados del Modelo RANDOM FOREST (COMPLETO - SMOTE)
Métrica      |      TRAIN     |     TEST   
-------------|----------------|---------------
Exactitud    |     0.9186     |    0.8672
Precisión    |     0.8892     |    0.4558
Sensibilidad |     0.9565     |    0.8779
F1-Score     |     0.9216     |    0.6001

--- TOP 2 | CV Score: 0.8993 ---
Hiperparámetros: {'n_estimators': 150, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 14, 'bootstrap': True}
Resultados del Modelo RANDOM FOREST (COMPLETO - SMOTE)
Métrica      |      TRAIN     |     TEST   
-------------|-----